# トレース・スコアリングとUC Trace Storageの仕組み

このノートブックでは、**span/logs/metricsの区別**と**DeltaTableとMLflow UIの関係**を
実際にデータを作成・確認しながら理解します。

## ストーリー

1. **Span を送信** — アプリからspanを送り、`_otel_spans` に格納されることを確認
2. **スコアを書き込み** — `mlflow.log_feedback()` で評価し、`_otel_logs` に格納されることを確認
3. **MLflow UI で確認** — spanはトレース詳細ビュー、スコアはAssessmentsパネルに表示
4. **Delta Table をSQLで分析** — レイテンシ、スコア集計、エラー分析など

## 前提条件
- MLflow 3.x がインストールされていること
- Unity Catalog trace storage が設定済みであること
- SQL Warehouse が指定されていること

# Part 1: セットアップ

## UC Trace Storage のテーブル命名ルール

UC trace storage を有効にすると、トレースデータは以下の4つの Delta Table に格納されます。  
各テーブル名の先頭には `table_prefix` が付きます。

```
<catalog>.<schema>.<table_prefix>_otel_spans
<catalog>.<schema>.<table_prefix>_otel_logs
<catalog>.<schema>.<table_prefix>_otel_annotations
<catalog>.<schema>.<table_prefix>_otel_metrics
```

`table_prefix` は `mlflow.set_experiment()` の `trace_location` で指定できます。  
省略した場合は実験IDが自動的に使われます。

| 指定方法 | 例 | 作成されるテーブル名 |
|---------|---|-------------------|
| 明示的に指定: `table_prefix="llmops_demo"` | `llmops_demo` | `main.mlflow_traces.llmops_demo_otel_spans` |
| 省略（実験IDが自動で使われる） | `12345`（実験ID） | `main.mlflow_traces.12345_otel_spans` |

このノートブックでは `table_prefix` を明示的に指定します。

## UC trace storage では独立エクスペリメントを使用する

通常のDatabricksノートブックでは、`mlflow.set_experiment()` を呼ばなくてもノートブックエクスペリメントが自動的に使われます。

**しかし、UC trace storage を使う場合は `trace_location` の指定が必要であり、そのためには `mlflow.set_experiment()` を明示的に呼ぶ必要があります。** このAPIは `experiment_name` が必須パラメータのため、独立エクスペリメントとして実験名を指定します。

この方式には利点もあります。複数のノートブックから同じ実験にトレースを記録でき、エクスペリメントの管理が明確になります。

In [0]:
%pip install "mlflow>=3.11"
dbutils.library.restartPython()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 36.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 54.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 52.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 611.4/611.4 kB 15.3 MB/s eta 0:00:00
  Attempting uninstall: blinker
    Found existing installation: blinker 1.7.0
    Not uninstalling blinker at /usr/lib/python3/dist-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-cb9b986c-343b-432b-bbde-66df3302755e
    Can't uninstall 'blinker'. No files were found to uninstall.
  Attempting uninstall: mlflow-skinny
    Found existing installation: mlflow-skinny 3.8.1
    Not uninstalling mlflow-skinny at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-cb9b986c-343b-432b-bbde-66df3302755e
    Can't uninstall 'm

In [0]:
import os
import mlflow
from mlflow.entities.trace_location import UnityCatalog
from mlflow.entities import AssessmentSource

mlflow.set_tracking_uri("databricks")

# UC trace storage の保存先
CATALOG_NAME = "takaakiyayoi_catalog"
SCHEMA_NAME = "mlflow_llmops"
TABLE_PREFIX_NAME = "llmops_demo"

# UC trace storage のクエリに使用する SQL Warehouse
os.environ["MLFLOW_TRACING_SQL_WAREHOUSE_ID"] = "bec52b183a4cfe2a"

# UC trace storage を使うには mlflow.set_experiment() の明示呼び出しが必要
# そのため独立エクスペリメントとして実験名を指定する
EXPERIMENT_NAME = "/Users/takaaki.yayoi@databricks.com/llmops_demo"
mlflow.set_experiment(
    experiment_name=EXPERIMENT_NAME,
    trace_location=UnityCatalog(
        catalog_name=CATALOG_NAME,
        schema_name=SCHEMA_NAME,
        table_prefix=TABLE_PREFIX_NAME,
    ),
)

# 後続のSQLクエリで使うテーブル名のプレフィックス
# 作成されるテーブル:
#   takaakiyayoi_catalog.mlflow_llmops.llmops_demo_otel_spans
#   takaakiyayoi_catalog.mlflow_llmops.llmops_demo_otel_logs
#   takaakiyayoi_catalog.mlflow_llmops.llmops_demo_otel_annotations
#   takaakiyayoi_catalog.mlflow_llmops.llmops_demo_otel_metrics
TABLE_PREFIX = f"{CATALOG_NAME}.{SCHEMA_NAME}.{TABLE_PREFIX_NAME}"

print(f"Experiment: {EXPERIMENT_NAME}")
print(f"テーブル例: {TABLE_PREFIX}_otel_spans")

Experiment: /Users/takaaki.yayoi@databricks.com/llmops_demo
テーブル例: takaakiyayoi_catalog.mlflow_llmops.llmops_demo_otel_spans


# Part 2: Span の送信

アプリが出力すべき唯一のOTELシグナルは **span** です。  
`@mlflow.trace` デコレータでspanを生成します。

```
生成されるspan階層:

agent_pipeline (AGENT)       ← root span
  ├── retrieve_context (RETRIEVER)
  └── generate_response (LLM)
```

> **Koog Frameworkがspansのみ対応でも問題ありません。**  
> root spanがあればMLflow experimentにtraceが表示されます。

In [0]:
@mlflow.trace(name="retrieve_context", span_type="RETRIEVER")
def retrieve_context(query: str) -> str:
    """検索ステップをシミュレート"""
    return f"'{query}' に関連するコンテキスト情報"


@mlflow.trace(name="generate_response", span_type="LLM")
def generate_response(query: str, context: str) -> str:
    """LLM呼び出しをシミュレート"""
    return f"回答: {query} について — {context} に基づく回答です。"


@mlflow.trace(name="agent_pipeline", span_type="AGENT")
def agent_pipeline(query: str) -> str:
    """エージェントのパイプライン全体（root span）"""
    context = retrieve_context(query)
    response = generate_response(query, context)
    return response


result = agent_pipeline("Databricksのセキュリティ機能について教えてください")
print(result)

回答: Databricksのセキュリティ機能について教えてください について — 'Databricksのセキュリティ機能について教えてください' に関連するコンテキスト情報 に基づく回答です。


Trace(trace_id=trace:/takaakiyayoi_catalog.mlflow_llmops.llmops_demo/a34fc94886aa115acc66fd46c3df0c4e)

In [0]:
# UC trace storage の場合、trace ID は URI 形式で返される
# 例: "trace:/catalog.schema.prefix/75edd04e18ea08a7d83ffb9f237c5d7d"
trace_id_full = mlflow.get_last_active_trace_id()

# MLflow API（log_feedback 等）には URI 形式をそのまま渡す
# SQLクエリには末尾のID部分のみを使う（テーブル内の trace_id カラムの形式）
trace_id_for_sql = trace_id_full.rsplit("/", 1)[-1]

print(f"Trace ID (API用): {trace_id_full}")
print(f"Trace ID (SQL用): {trace_id_for_sql}")

Trace ID (API用): trace:/takaakiyayoi_catalog.mlflow_llmops.llmops_demo/a34fc94886aa115acc66fd46c3df0c4e
Trace ID (SQL用): a34fc94886aa115acc66fd46c3df0c4e


### ここで MLflow UI を確認

該当experimentのトレース詳細ビューに、3つのspanが階層構造で表示されていることを確認してください。

# Part 3: スコアの書き込み（Assessment）

スコアは **assessment** としてtraceに紐付けます。  
書き込み後の反映先:

| 反映先 | 表示内容 |
|--------|----------|
| **MLflow UI — トレースリスト** | 各assessment名が独立カラムとして表示 |
| **MLflow UI — トレース詳細** | Assessmentsパネルに値とrationaleが表示 |
| **`_otel_logs` テーブル** | 構造化イベントとして格納（後のPartで確認） |

In [0]:
# 人間のフィードバック
mlflow.log_feedback(
    trace_id=trace_id_full,
    name="user_satisfaction",
    value=4.0,
    source=AssessmentSource(source_type="HUMAN", source_id="demo_user"),
    rationale="回答は正確だが、もう少し具体例が欲しい",
)

# 自動評価スコア
mlflow.log_feedback(
    trace_id=trace_id_full,
    name="relevance_score",
    value=0.85,
    source=AssessmentSource(source_type="CODE", source_id="relevance_scorer"),
    rationale="コンテキストと回答の関連性が高い",
)

# 正解との比較
mlflow.log_feedback(
    trace_id=trace_id_full,
    name="correctness",
    value=True,
    source=AssessmentSource(source_type="HUMAN", source_id="demo_user"),
    rationale="期待される回答と一致",
)

print("スコアを3件書き込みました")

スコアを3件書き込みました


### ここで MLflow UI を確認

- **トレースリスト**: `user_satisfaction`, `relevance_score`, `correctness` が独立カラムとして表示
- **トレース詳細 → Assessments パネル**: 各スコアの値と rationale が表示

# Part 4: Delta Table の中身を確認

ここからが**DeltaTableとMLflow UIの関連**の核心です。  
UC trace storageでは、MLflow UIが読み取るデータはすべてDelta Tableに格納されています。

Databricksは基盤テーブル（`_otel_spans` 等）のスキーマは変更される可能性があるため、  
**自動生成されるビュー（`_trace_unified`, `_trace_metadata`）経由でクエリすることを推奨**しています。

| クエリ対象 | 説明 |
|-----------|------|
| `<table_prefix>_trace_unified` | span + assessment + メタデータを結合したビュー（推奨） |
| `<table_prefix>_trace_metadata` | トレース単位のメタデータ・タグ・assessment |
| `<table_prefix>_otel_spans` 等 | 基盤テーブル（スキーマ変更の可能性あり、直接利用は非推奨） |

## 4-1. trace_unified ビューでトレース全体を確認

`_trace_unified` は**トレース単位**のビューです（1行 = 1トレース）。  
spanデータは `spans` カラム（配列）にネストされています。

In [0]:
# トレース全体の情報を確認
df_trace = spark.sql(f"""
    SELECT
        trace_id,
        request_time,
        state,
        execution_duration_ms,
        request,
        response
    FROM {TABLE_PREFIX}_trace_unified
    WHERE trace_id = '{trace_id_for_sql}'
""")

display(df_trace)

trace_id,request_time,state,execution_duration_ms,request,response
a34fc94886aa115acc66fd46c3df0c4e,2026-05-20T05:49:04.283Z,OK,324.312923,"{""query"":""Databricksのセキュリティ機能について教えてください""}",回答: Databricksのセキュリティ機能について教えてください について — 'Databricksのセキュリティ機能について教えてください' に関連するコンテキスト情報 に基づく回答です。


# spans カラムを展開して個別のspanを確認
df_spans = spark.sql(f"""
    SELECT
        s.span_id,
        s.parent_span_id,
        s.name,
        s.start_time_unix_nano,
        s.end_time_unix_nano,
        s.status.code AS status_code,
        s.attributes
    FROM {TABLE_PREFIX}_trace_unified,
    LATERAL VIEW explode(spans) AS s
    WHERE trace_id = '{trace_id_for_sql}'
    ORDER BY s.start_time_unix_nano
""")

display(df_spans)

**確認ポイント:**
- 1つ目のクエリ: トレース全体の情報（リクエスト、レスポンス、実行時間）
- 2つ目のクエリ: `spans` 配列を `explode` で展開し、3つのspan（agent_pipeline, retrieve_context, generate_response）が `parent_span_id` で階層構造を持っていることを確認

In [0]:
df_metadata = spark.sql(f"""
    SELECT
        trace_id,
        assessments,
        tags,
        trace_metadata
    FROM {TABLE_PREFIX}_trace_metadata
    WHERE trace_id = '{trace_id_for_sql}'
""")

display(df_metadata)

trace_id,assessments,tags,trace_metadata
a34fc94886aa115acc66fd46c3df0c4e,"List(List(a-2870dddc713f424995491fcd1c8c6283, a34fc94886aa115acc66fd46c3df0c4e, correctness, List(demo_user, HUMAN), 2026-05-20T05:49:05.435Z, 2026-05-20T05:49:05.435Z, null, List(true, null), 期待される回答と一致, Map(mlflow.source.source_id -> demo_user, mlflow.source.source_type -> HUMAN, mlflow.valid -> true), null, null, true), List(a-3418825a950d46a8a5bd940c92a07c46, a34fc94886aa115acc66fd46c3df0c4e, relevance_score, List(relevance_scorer, CODE), 2026-05-20T05:49:05.230Z, 2026-05-20T05:49:05.229Z, null, List(0.85, null), コンテキストと回答の関連性が高い, Map(mlflow.source.source_id -> relevance_scorer, mlflow.source.source_type -> CODE, mlflow.valid -> true), null, null, true), List(a-35180b60950b4d91a86f35afacdd38f6, a34fc94886aa115acc66fd46c3df0c4e, user_satisfaction, List(demo_user, HUMAN), 2026-05-20T05:49:05.133Z, 2026-05-20T05:49:05.121Z, null, List(4.0, null), 回答は正確だが、もう少し具体例が欲しい, Map(mlflow.source.source_id -> demo_user, mlflow.source.source_type -> HUMAN, mlflow.valid -> true), null, null, true))",Map(mlflow.traceName -> agent_pipeline),"{""mlflow.databricks.notebook.commandID"":""1779255006297_5089311364875628165_689379ba927a4c2faa29d56d2c647567"",""mlflow.databricks.notebookID"":""150864305197143"",""mlflow.databricks.notebookPath"":""/Users/takaaki.yayoi@databricks.com/20260520_mlflow_llmops/tracing_scoring_and_tables"",""mlflow.databricks.webappURL"":""https://e2-demo-tokyo.cloud.databricks.com"",""mlflow.databricks.workspaceID"":""5099015744649857"",""mlflow.databricks.workspaceURL"":""https://e2-demo-tokyo.cloud.databricks.com"",""mlflow.source.name"":""/Users/takaaki.yayoi@databricks.com/20260520_mlflow_llmops/tracing_scoring_and_tables"",""mlflow.source.type"":""NOTEBOOK"",""mlflow.trace.sizeBytes"":""5991"",""mlflow.trace.sizeStats"":""{\""total_size_bytes\"": 5991, \""num_spans\"": 3, \""max\"": 1377, \""p25\"": 1026, \""p50\"": 1133, \""p75\"": 1255}"",""mlflow.traceInputs"":""{\""query\"": \""Databricksのセキュリティ機能について教えてください\""}"",""mlflow.traceOutputs"":""\""回答: Databricksのセキュリティ機能について教えてください について — 'Databricksのセキュリティ機能について教えてください' に関連するコンテキスト情報 に基づく回答です。\"""",""mlflow.trace_schema.version"":""4"",""mlflow.user"":""spark-cb9b986c-343b-432b-bbde-66""}"


**確認ポイント:**
- `assessments` カラムに `mlflow.log_feedback()` で書き込んだ3件のアセスメントが含まれている
- **アプリは span のみ出力**している。アセスメントはMLflow API呼び出し時にプラットフォームが格納する
- これがMLflow UIのトレースリストでAssessmentカラムとして表示される

## 4-3. 基盤テーブルの行数を確認

In [0]:
# 基盤テーブルの行数を確認（スキーマは変更される可能性があるため参考程度）
for table in ["_otel_spans", "_otel_logs", "_otel_annotations", "_otel_metrics"]:
    count = spark.sql(f"SELECT COUNT(*) AS cnt FROM {TABLE_PREFIX}{table}").first()["cnt"]
    print(f"{TABLE_PREFIX}{table}: {count} 行")

takaakiyayoi_catalog.mlflow_llmops.llmops_demo_otel_spans: 18 行
takaakiyayoi_catalog.mlflow_llmops.llmops_demo_otel_logs: 0 行
takaakiyayoi_catalog.mlflow_llmops.llmops_demo_otel_annotations: 25 行
takaakiyayoi_catalog.mlflow_llmops.llmops_demo_otel_metrics: 0 行


**確認ポイント:**
- `_otel_spans`: span が格納されている
- `_otel_logs`: アセスメントが格納されている
- `_otel_metrics`: カスタム計装を行っていないため空（0件）。MLflow UIにも非表示
- `_otel_annotations`: MLflow固有のメタデータ

**重要:** MLflow UIはこれらのDelta Tableを直接読み取っています。SQLで変更すればUIにも反映されます（スキーマの整合性維持が必要）。

# Part 5: SQLによるトレース分析

UC trace storageを使うことで、MLflow UIでは難しい分析がSQLで可能になります。

## 5-1. 直近のトレース一覧

In [0]:
# 直近のトレース一覧（トレース単位）
df = spark.sql(f"""
    SELECT
        trace_id,
        request_time,
        state,
        execution_duration_ms,
        request,
        response
    FROM {TABLE_PREFIX}_trace_unified
    ORDER BY request_time DESC
    LIMIT 20
""")

display(df)

trace_id,request_time,state,execution_duration_ms,request,response
a34fc94886aa115acc66fd46c3df0c4e,2026-05-20T05:49:04.283Z,OK,324.312923,"{""query"":""Databricksのセキュリティ機能について教えてください""}",回答: Databricksのセキュリティ機能について教えてください について — 'Databricksのセキュリティ機能について教えてください' に関連するコンテキスト情報 に基づく回答です。
379f616d9a1f2c9a398c58e567794ede,2026-05-20T05:44:23.602Z,OK,355.541178,"{""query"":""Databricksのセキュリティ機能について教えてください""}",回答: Databricksのセキュリティ機能について教えてください について — 'Databricksのセキュリティ機能について教えてください' に関連するコンテキスト情報 に基づく回答です。
75edd04e18ea08a7d83ffb9f237c5d7d,2026-05-20T05:35:26.546Z,OK,449.90984,"{""query"":""Databricksのセキュリティ機能について教えてください""}",回答: Databricksのセキュリティ機能について教えてください について — 'Databricksのセキュリティ機能について教えてください' に関連するコンテキスト情報 に基づく回答です。
df35142d83a4fb8caaa0c4558ada2188,2026-05-20T05:23:48.130Z,OK,159.175981,"{""query"":""Databricksのセキュリティ機能について教えてください""}",回答: Databricksのセキュリティ機能について教えてください について — 'Databricksのセキュリティ機能について教えてください' に関連するコンテキスト情報 に基づく回答です。
492d7f7f4cd9e19a9ebd65f306282ccb,2026-05-20T05:20:31.534Z,OK,168.577527,"{""query"":""Databricksのセキュリティ機能について教えてください""}",回答: Databricksのセキュリティ機能について教えてください について — 'Databricksのセキュリティ機能について教えてください' に関連するコンテキスト情報 に基づく回答です。
284be6d222cc82f60b3414c7dd44a167,2026-05-20T05:10:54.142Z,OK,269.544673,"{""query"":""Databricksのセキュリティ機能について教えてください""}",回答: Databricksのセキュリティ機能について教えてください について — 'Databricksのセキュリティ機能について教えてください' に関連するコンテキスト情報 に基づく回答です。


## 5-2. スコア別の統計

In [0]:
# アセスメント一覧（trace_metadata ビューを使用、軽量）
df = spark.sql(f"""
    SELECT
        trace_id,
        assessments
    FROM {TABLE_PREFIX}_trace_metadata
    WHERE assessments IS NOT NULL
    LIMIT 50
""")

display(df)

trace_id,assessments
75edd04e18ea08a7d83ffb9f237c5d7d,"List(List(a-400e749476ab47ed8d519a3ea3f55ef7, 75edd04e18ea08a7d83ffb9f237c5d7d, user_satisfaction, List(demo_user, HUMAN), 2026-05-20T05:35:27.575Z, 2026-05-20T05:35:27.574Z, null, List(4.0, null), 回答は正確だが、もう少し具体例が欲しい, Map(mlflow.source.source_id -> demo_user, mlflow.source.source_type -> HUMAN, mlflow.valid -> true), null, null, true), List(a-67b86e9fe5fa4556ba62f3c8ec7c55cb, 75edd04e18ea08a7d83ffb9f237c5d7d, relevance_score, List(relevance_scorer, CODE), 2026-05-20T05:35:27.799Z, 2026-05-20T05:35:27.799Z, null, List(0.85, null), コンテキストと回答の関連性が高い, Map(mlflow.source.source_id -> relevance_scorer, mlflow.source.source_type -> CODE, mlflow.valid -> true), null, null, true), List(a-e6ab769e7d3c4bb694c529ff071c8e79, 75edd04e18ea08a7d83ffb9f237c5d7d, correctness, List(demo_user, HUMAN), 2026-05-20T05:35:28.000Z, 2026-05-20T05:35:27.999Z, null, List(true, null), 期待される回答と一致, Map(mlflow.source.source_id -> demo_user, mlflow.source.source_type -> HUMAN, mlflow.valid -> true), null, null, true))"
a34fc94886aa115acc66fd46c3df0c4e,"List(List(a-2870dddc713f424995491fcd1c8c6283, a34fc94886aa115acc66fd46c3df0c4e, correctness, List(demo_user, HUMAN), 2026-05-20T05:49:05.435Z, 2026-05-20T05:49:05.435Z, null, List(true, null), 期待される回答と一致, Map(mlflow.source.source_id -> demo_user, mlflow.source.source_type -> HUMAN, mlflow.valid -> true), null, null, true), List(a-3418825a950d46a8a5bd940c92a07c46, a34fc94886aa115acc66fd46c3df0c4e, relevance_score, List(relevance_scorer, CODE), 2026-05-20T05:49:05.230Z, 2026-05-20T05:49:05.229Z, null, List(0.85, null), コンテキストと回答の関連性が高い, Map(mlflow.source.source_id -> relevance_scorer, mlflow.source.source_type -> CODE, mlflow.valid -> true), null, null, true), List(a-35180b60950b4d91a86f35afacdd38f6, a34fc94886aa115acc66fd46c3df0c4e, user_satisfaction, List(demo_user, HUMAN), 2026-05-20T05:49:05.133Z, 2026-05-20T05:49:05.121Z, null, List(4.0, null), 回答は正確だが、もう少し具体例が欲しい, Map(mlflow.source.source_id -> demo_user, mlflow.source.source_type -> HUMAN, mlflow.valid -> true), null, null, true))"
379f616d9a1f2c9a398c58e567794ede,"List(List(a-1b85b4aaedee4aa0bf0768c34ce2f5b7, 379f616d9a1f2c9a398c58e567794ede, test_full_id, List(demo_user, HUMAN), 2026-05-20T05:44:24.634Z, 2026-05-20T05:44:24.623Z, null, List(1.0, null), , Map(mlflow.source.source_id -> demo_user, mlflow.source.source_type -> HUMAN, mlflow.valid -> true), null, null, true))"
492d7f7f4cd9e19a9ebd65f306282ccb,"List(List(a-1c526db7a4194aa0806af0dd803f8e06, 492d7f7f4cd9e19a9ebd65f306282ccb, correctness, List(demo_user, HUMAN), 2026-05-20T05:20:42.848Z, 2026-05-20T05:20:42.847Z, null, List(true, null), 期待される回答と一致, Map(mlflow.source.source_id -> demo_user, mlflow.source.source_type -> HUMAN, mlflow.valid -> true), null, null, true), List(a-4241426633754b9f828ffcf23e85c0cc, 492d7f7f4cd9e19a9ebd65f306282ccb, user_satisfaction, List(demo_user, HUMAN), 2026-05-20T05:20:42.530Z, 2026-05-20T05:20:42.528Z, null, List(4.0, null), 回答は正確だが、もう少し具体例が欲しい, Map(mlflow.source.source_id -> demo_user, mlflow.source.source_type -> HUMAN, mlflow.valid -> true), null, null, true), List(a-9b91ee3fd26640b2958fd5fee87f7613, 492d7f7f4cd9e19a9ebd65f306282ccb, relevance_score, List(relevance_scorer, CODE), 2026-05-20T05:20:42.672Z, 2026-05-20T05:20:42.671Z, null, List(0.85, null), コンテキストと回答の関連性が高い, Map(mlflow.source.source_id -> relevance_scorer, mlflow.source.source_type -> CODE, mlflow.valid -> true), null, null, true))"
df35142d83a4fb8caaa0c4558ada2188,"List(List(a-03ca4bcfb85d480c997596765ecb957d, df35142d83a4fb8caaa0c4558ada2188, relevance_score, List(relevance_scorer, CODE), 2026-05-20T05:24:14.421Z, 2026-05-20T05:24:14.421Z, null, List(0.85, null), コンテキストと回答の関連性が高い, Map(mlflow.source.source_id -> relevance_scorer, mlflow.source.source_type -> CODE, mlflow.valid -> true), null, null, true), List(a-772a7e84f41a4faa92b77383762918c4, df35142d83a4fb8caaa0c4558ada2188, correctness, List(demo_user, HUMAN), 2026-05-20T05:24:14.647Z, 20

## 5-3. レイテンシ分析（スパン種別ごと）

In [0]:
# レイテンシ分析（トレース単位）
df = spark.sql(f"""
    SELECT
        COUNT(*) AS trace_count,
        CAST(AVG(execution_duration_ms) AS DECIMAL(10,1)) AS avg_ms,
        CAST(PERCENTILE(execution_duration_ms, 0.5) AS DECIMAL(10,1)) AS p50_ms,
        CAST(PERCENTILE(execution_duration_ms, 0.95) AS DECIMAL(10,1)) AS p95_ms,
        CAST(MAX(execution_duration_ms) AS DECIMAL(10,1)) AS max_ms
    FROM {TABLE_PREFIX}_trace_unified
""")

display(df)

trace_count,avg_ms,p50_ms,p95_ms,max_ms
6,287.8,296.9,426.3,449.9


## 5-4. 日別トレース件数の推移

In [0]:
# 日別トレース件数の推移
df = spark.sql(f"""
    SELECT
        DATE(request_time) AS date,
        COUNT(*) AS trace_count
    FROM {TABLE_PREFIX}_trace_unified
    GROUP BY DATE(request_time)
    ORDER BY date DESC
    LIMIT 30
""")

display(df)

date,trace_count
2026-05-20,6


## 5-5. エラーが発生したトレースの特定

まずエラーを発生させるトレースを作成し、その後 `state = 'ERROR'` で検索します。

In [0]:
# エラーを発生させるトレースを作成
@mlflow.trace(name="failing_pipeline", span_type="AGENT")
def failing_pipeline(query: str) -> str:
    raise ValueError(f"処理に失敗しました: {query}")

try:
    failing_pipeline("エラーテスト")
except ValueError:
    pass

import time
print("エラートレースを作成しました。テーブルへの反映を待ちます...")
time.sleep(10)

# エラーが発生したトレースを検索
df = spark.sql(f"""
    SELECT
        trace_id,
        request_time,
        state,
        execution_duration_ms,
        request
    FROM {TABLE_PREFIX}_trace_unified
    WHERE state = 'ERROR'
    ORDER BY request_time DESC
    LIMIT 20
""")

display(df)

エラートレースを作成しました。テーブルへの反映を待ちます...


trace_id,request_time,state,execution_duration_ms,request
534e6dbb5ab7aca8b33a29208bfed270,2026-05-20T05:49:32.904Z,ERROR,2.092234,"{""query"":""エラーテスト""}"


Trace(trace_id=trace:/takaakiyayoi_catalog.mlflow_llmops.llmops_demo/534e6dbb5ab7aca8b33a29208bfed270)

# まとめ

## データフローの全体像

```
アプリ (@mlflow.trace)
  │
  │ span を送信
  ▼
_otel_spans  ──────→  MLflow UI: トレース詳細ビュー（span階層）

mlflow.log_feedback()
  │
  │ プラットフォームが OTEL log 形式に変換
  ▼
_otel_logs   ──────→  MLflow UI: トレースリスト（Assessmentカラム）
                                  トレース詳細（Assessmentsパネル）

カスタム計装（任意）
  │
  ▼
_otel_metrics ─ ✕ →  MLflow UI には非表示（SQLで直接参照が必要）
```

## SQLクエリのポイント

| 項目 | 内容 |
|------|------|
| 推奨クエリ先 | `_trace_unified` ビュー（1行=1トレース）または `_trace_metadata` ビュー |
| spanの参照 | `spans` カラム（LIST&lt;STRUCT&gt;）を `explode` で展開 |
| assessmentの参照 | `assessments` カラム（LIST&lt;STRUCT&gt;）。`_trace_metadata` が軽量 |
| 基盤テーブル直接 | `_otel_spans` 等はスキーマ変更の可能性あり。直接利用は非推奨 |

## 要点

| 項目 | 内容 |
|------|------|
| アプリが出力するもの | **span のみ** |
| `_otel_logs` に書き込むもの | `mlflow.log_feedback()` の結果。プラットフォームがOTEL形式に変換して格納 |
| `_otel_metrics` | カスタム計装しなければ空。MLflow UIにも非表示 |
| DeltaTable と MLflow UI | **MLflow UIはDelta Tableを直接読み取る。** SQLで変更すればUIにも反映される |